# Batch Model Monitoring Demo - Simulate Drift

Generates 7 days of progressively drifted scoring data to trigger drift alerts:

| Day | Drift Applied |
|-----|---------------|
| 1 (baseline) | Clean — matches training distribution |
| 2-3 | `monthly_charges` mean increases 15-30% |
| 4-5 | `tenure_months` skews shorter + charges continue to rise |
| 6-7 | Combined: age shifts younger, usage drops, prediction scores climb |

After running, wait 1-2 monitor refresh cycles, then run `06_observe_and_query.ipynb`.

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np
import uuid
from datetime import datetime, timedelta

session = get_active_session()
session.use_schema("ML_DEMOS.BATCH_MONITORING")
session.use_warehouse("ML_DEMO_WH")

np.random.seed(123)

PLAN_TYPES = ["BASIC", "BASIC", "BASIC", "PREMIUM", "PREMIUM", "ENTERPRISE"]
CONTRACT_TYPES = ["MONTH_TO_MONTH", "MONTH_TO_MONTH", "MONTH_TO_MONTH", "ONE_YEAR", "ONE_YEAR", "TWO_YEAR"]
NUM_PER_DAY = 2000
BASE_DATE = datetime(2025, 1, 16)  # Day after initial scoring

print(f"Will generate {NUM_PER_DAY} records per day for 7 days")
print(f"Date range: {BASE_DATE.strftime('%Y-%m-%d')} to {(BASE_DATE + timedelta(days=6)).strftime('%Y-%m-%d')}")

In [ ]:
def generate_day(day_offset: int) -> pd.DataFrame:
    """Generate one day of scoring data with progressive drift."""
    n = NUM_PER_DAY
    prediction_ts = BASE_DATE + timedelta(days=day_offset)

    # --- Base distributions (matching training data) ---
    tenure = np.random.randint(1, 72, size=n).astype(float)
    monthly_charges = np.random.uniform(24, 130, size=n)
    age = np.random.randint(18, 70, size=n).astype(float)
    support_tickets = np.random.choice([0,0,0,1,1,2,3,5,8], size=n)
    days_login = np.random.choice([0,1,1,2,3,5,7,14,30,60], size=n)
    usage_hours = np.clip(np.random.normal(45, 20, size=n), 0.5, None)

    # --- Apply progressive drift ---
    if day_offset >= 1:  # Days 2-3: charges drift up
        charge_multiplier = 1.0 + (0.15 * day_offset)
        monthly_charges *= charge_multiplier

    if day_offset >= 3:  # Days 4-5: tenure skews shorter
        tenure = np.clip(tenure * 0.5, 1, 72)

    if day_offset >= 5:  # Days 6-7: age younger + usage drops
        age = np.clip(age - 15, 18, 70)
        usage_hours = np.clip(usage_hours * 0.5, 0.5, None)

    # Compute synthetic prediction scores (higher with more drift)
    base_score = 0.3
    score_boost = 0.05 * day_offset
    prediction_score = np.clip(
        np.random.beta(2 + day_offset * 0.5, 5 - day_offset * 0.3, size=n) + score_boost,
        0.01, 0.99
    )

    total_charges = monthly_charges * tenure * np.random.uniform(0.9, 1.0, size=n)
    plan_type = np.random.choice(PLAN_TYPES, size=n)
    contract_type = np.random.choice(CONTRACT_TYPES, size=n)

    return pd.DataFrame({
        "ID": [str(uuid.uuid4()) for _ in range(n)],
        "CUSTOMER_ID": [f"C{np.random.randint(1, 2001):05d}" for _ in range(n)],
        "PREDICTION_TS": prediction_ts.strftime("%Y-%m-%d %H:%M:%S"),
        "TENURE_MONTHS": tenure.astype(int),
        "MONTHLY_CHARGES": np.round(monthly_charges, 2),
        "TOTAL_CHARGES": np.round(total_charges, 2),
        "NUM_SUPPORT_TICKETS": support_tickets.astype(int),
        "DAYS_SINCE_LAST_LOGIN": days_login.astype(int),
        "AVG_MONTHLY_USAGE_HOURS": np.round(usage_hours, 1),
        "AGE": age.astype(int),
        "CONTRACT_TYPE": contract_type,
        "PLAN_TYPE": plan_type,
        "PREDICTION_SCORE": np.round(prediction_score, 4),
    })

# Preview one day's drift characteristics
sample = generate_day(6)
print("Day 7 (max drift) statistics:")
print(sample[["MONTHLY_CHARGES", "TENURE_MONTHS", "AGE", "AVG_MONTHLY_USAGE_HOURS", "PREDICTION_SCORE"]].describe())

In [ ]:
# Insert all 7 days into SCORING_DATA (append mode)
total_inserted = 0

for day in range(7):
    day_df = generate_day(day)
    sp_df = session.create_dataframe(day_df)
    sp_df.write.mode("append").save_as_table("SCORING_DATA")
    total_inserted += len(day_df)
    date_str = (BASE_DATE + timedelta(days=day)).strftime("%Y-%m-%d")
    avg_score = day_df["PREDICTION_SCORE"].mean()
    avg_charges = day_df["MONTHLY_CHARGES"].mean()
    print(f"  Day {day+1} ({date_str}): {len(day_df)} rows | avg_score={avg_score:.3f} | avg_charges=${avg_charges:.0f}")

print(f"\nTotal inserted: {total_inserted:,} rows")

In [ ]:
%%sql -r df_summary
-- Verify: row counts per day
SELECT
    prediction_ts::DATE AS scoring_date,
    COUNT(*) AS row_count,
    ROUND(AVG(prediction_score), 3) AS avg_score,
    ROUND(AVG(monthly_charges), 1) AS avg_charges,
    ROUND(AVG(tenure_months), 1) AS avg_tenure
FROM SCORING_DATA
GROUP BY 1
ORDER BY 1;

## Next Steps

The drifted data is now in SCORING_DATA. The model monitor will detect these shifts
on its next refresh cycle (configured at 1 hour intervals).

**To see results sooner**, you can manually trigger a refresh by suspending and resuming:
```sql
ALTER MODEL MONITOR CHURN_MONITOR SUSPEND;
ALTER MODEL MONITOR CHURN_MONITOR RESUME;
```

Then run `06_observe_and_query.ipynb` to query the drift metrics programmatically,
or navigate to **Snowsight → AI & ML → Models → CHURN_PREDICTOR → Monitors**
to see the visual dashboard.